In [ ]:
import zipfile
from google.colab import drive
drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/high_plains_quifer.zip'
extract_path = '/content/ogallala_shp'

# Extract the zip file
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Files extracted to:", extract_path)

!pip install -q xee xarray netcdf4 geopandas pyproj dask h5netcdf

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files extracted to: /content/ogallala_shp
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.3 MB/s eta 0:00:00


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# ============================================================================
# SMAP L4 (SPL4SMGP) Extractor — Batch Validation & Redownload Logic
# ============================================================================
!pip -q install earthaccess geopandas shapely h5py netCDF4 2>&1 | tail -2

import os, re, gc, glob, time, shutil, logging, hashlib
from datetime import datetime, timezone, timedelta
import numpy as np
import h5py
import netCDF4 as nc4
import geopandas as gpd
from shapely.vectorized import contains as vec_contains
import earthaccess

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)-7s | %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('smap')

# ─── CONFIGURATION ──────────────────────────────────────────────────────────
SHORT_NAME      = "SPL4SMGP"
SHAP_PATH       = '/content/ogallala_shp/high_plains_quifer/hp_bound2010.shp'
DRIVE_OUT_DIR   = '/content/drive/MyDrive/SMAP_L4_Ogallala'
LOCAL_WORK      = '/content/smap_raw'
LOCAL_NC        = '/content/smap_nc'

SINGLE_FILE_MODE = True

VARIABLES = [
    "baseflow_flux",
    "depth_to_water_table_from_surface",
    "land_evapotranspiration_flux",
    "sm_rootzone",
    "sm_rootzone_pctl"
]
FILL_VALUE      = np.float32(-9999.0)
BATCH_DATES     = 150
FLUSH_DATES     = 25
BACKUP_DATES    = 75
MAX_RETRIES     = 4         # Max attempts to redownload corrupted files
N_THREADS       = 12
COMPRESS_LEVEL  = 3

# Regex matches: SMAP_L4_SM_gph_20150827T012300_Vv8010_001.h5
SMAP_RE = re.compile(r'SMAP_L4_SM_gph_(\d{8})T(\d{4})0.*\.h5$')
# Regex matches: SMAP_ogallala_3hr.nc
RUNTIME_BKUP_RE = re.compile(r'SMAP_ogallala_(\d+)hr\.nc$')

for d in (LOCAL_WORK, LOCAL_NC, DRIVE_OUT_DIR):
    os.makedirs(d, exist_ok=True)

# ─── UTILITIES ──────────────────────────────────────────────────────────────
def parse_dt(filename):
    m = SMAP_RE.search(os.path.basename(filename))
    if not m:
        return None
    return datetime.strptime(m.group(1) + m.group(2), '%Y%m%d%H%M')

def get_latest_drive_backup(drive_dir):
    """Finds the most recent runtime backup or 75-day backup in Drive."""
    hr_files = glob.glob(os.path.join(drive_dir, 'SMAP_ogallala_*hr.nc'))
    latest_hr_file = None
    max_hr = -1

    if hr_files:
        for f in hr_files:
            match = RUNTIME_BKUP_RE.search(os.path.basename(f))
            if match:
                hr_val = int(match.group(1))
                if hr_val > max_hr:
                    max_hr = hr_val
                    latest_hr_file = f

    full_file = os.path.join(drive_dir, 'SPL4SMGP_Ogallala_FULL.nc')

    # Compare modification times if both exist
    candidates = []
    if latest_hr_file and os.path.exists(latest_hr_file):
        candidates.append((os.path.getmtime(latest_hr_file), latest_hr_file))
    if os.path.exists(full_file):
        candidates.append((os.path.getmtime(full_file), full_file))

    if candidates:
        candidates.sort(reverse=True) # Sort by mod time descending
        return candidates[0][1]
    return None

def load_roi(shp_path, deg=0.1):
    gdf = gpd.read_file(shp_path)
    if gdf.crs is None or gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(epsg=4326)
    roi = gdf.geometry.unary_union
    return roi.simplify(deg, preserve_topology=True)

def build_mask(sample_h5, roi):
    with h5py.File(sample_h5, 'r') as f:
        cell_lat = f['cell_lat'][:]
        cell_lon = f['cell_lon'][:]

    minx, miny, maxx, maxy = roi.bounds
    bbox_mask = (cell_lat >= miny) & (cell_lat <= maxy) & (cell_lon >= minx) & (cell_lon <= maxx)
    rows = np.where(np.any(bbox_mask, axis=1))[0]
    cols = np.where(np.any(bbox_mask, axis=0))[0]
    r0, r1 = int(rows[0]), int(rows[-1]) + 1
    c0, c1 = int(cols[0]), int(cols[-1]) + 1

    lat_sub = cell_lat[r0:r1, c0:c1]
    lon_sub = cell_lon[r0:r1, c0:c1]
    mask_2d = vec_contains(roi, lon_sub, lat_sub)

    log.info(f"ROI subset: {lat_sub.shape}, Pixels in ROI: {mask_2d.sum()}")
    return slice(r0, r1), slice(c0, c1), mask_2d, lat_sub, lon_sub

def create_nc(nc_path, lat_2d, lon_2d, mask_2d):
    ny, nx = lat_2d.shape
    ds = nc4.Dataset(nc_path, 'w', format='NETCDF4')
    ds.createDimension('time', None)
    ds.createDimension('y', ny)
    ds.createDimension('x', nx)

    t_var = ds.createVariable('time', 'f8', ('time',))
    t_var.units = 'hours since 2000-01-01 00:00:00 UTC'
    t_var.calendar = 'proleptic_gregorian'

    lat_var = ds.createVariable('lat', 'f4', ('y', 'x'), zlib=True, complevel=1)
    lat_var[:] = lat_2d; lat_var.units = 'degrees_north'; lat_var.standard_name = 'latitude'

    lon_var = ds.createVariable('lon', 'f4', ('y', 'x'), zlib=True, complevel=1)
    lon_var[:] = lon_2d; lon_var.units = 'degrees_east'; lon_var.standard_name = 'longitude'

    m_var = ds.createVariable('roi_mask', 'i1', ('y', 'x'), zlib=True, complevel=1)
    m_var[:] = mask_2d.astype(np.int8)

    chunk_3d = (8, min(256, ny), min(256, nx))
    for vname in VARIABLES:
        ds.createVariable(vname, 'f4', ('time','y','x'), fill_value=FILL_VALUE,
                          zlib=True, complevel=COMPRESS_LEVEL, chunksizes=chunk_3d)
        ds[vname].coordinates = 'lat lon'

    crs_var = ds.createVariable('crs', 'i4')
    crs_var.grid_mapping_name = 'lambert_cylindrical_equal_area'
    crs_var.standard_parallel = 30.0; crs_var.longitude_of_central_meridian = 0.0
    crs_var.crs_wkt = 'PROJCRS["EASE-Grid 2.0 Global",BASEGEOGCRS["WGS 84",...METHOD["Lambert Cylindrical Equal Area"]]'

    ds.Conventions = 'CF-1.8'
    ds.title = 'SMAP L4 SM (SPL4SMGP) — Ogallala ROI Subset'
    ds.close()

def validate_h5(filepath):
    """Check if an HDF5 file is readable and has the expected structure."""
    try:
        with h5py.File(filepath, 'r') as f:
            if 'cell_lat' not in f or 'Geophysical_Data' not in f: return False
            gd = f['Geophysical_Data']
            for v in VARIABLES:
                if v not in gd: return False
                _ = gd[v][:1]  # Force read to catch silent corruption
        return True
    except Exception:
        return False

def extract_granule(h5_path, row_slice, col_slice, mask_2d):
    with h5py.File(h5_path, 'r') as f:
        gd = f['Geophysical_Data']
        data = {}
        for v in VARIABLES:
            arr = gd[v][row_slice, col_slice].astype(np.float32)
            arr[~mask_2d] = FILL_VALUE
            data[v] = arr
    return data

def get_md5(filepath):
    h = hashlib.md5()
    with open(filepath, 'rb') as f:
        while chunk := f.read(10485760): h.update(chunk)
    return h.hexdigest()

def robust_drive_copy(src, dst):
    src_md5 = get_md5(src)
    for attempt in range(1, 4):
        try:
            with open(src, 'rb') as fs, open(dst, 'wb') as fd:
                shutil.copyfileobj(fs, fd, length=10485760)
                fd.flush(); os.fsync(fd.fileno())
            if os.path.getsize(src) == os.path.getsize(dst) and src_md5 == get_md5(dst):
                log.info(f"Drive copy verified ({os.path.getsize(dst)/1048576:.1f} MiB)")
                return True
            raise IOError("Size or MD5 mismatch")
        except Exception as e:
            log.error(f"Drive copy attempt {attempt} failed: {e}")
            if os.path.exists(dst): os.remove(dst)
            time.sleep(5 * attempt)
    raise RuntimeError(f"Drive copy failed after 3 attempts: {e}")

# ─── PIPELINE ───────────────────────────────────────────────────────────────
def run_pipeline():
    earthaccess.login(strategy="interactive")
    roi = load_roi(SHAP_PATH)

    log.info("Fetching sample granule for grid...")
    res = earthaccess.search_data(short_name=SHORT_NAME, temporal=('2020-01-01', '2020-01-01'), count=1)
    sample = str(earthaccess.download(res, LOCAL_WORK)[0])
    row_slice, col_slice, mask_2d, lat_sub, lon_sub = build_mask(sample, roi)
    os.remove(sample)

    # ─── RESUME LOGIC ───
    local_nc = os.path.join(LOCAL_NC, 'SPL4SMGP_Ogallala_FULL.nc')
    if SINGLE_FILE_MODE:
        if not os.path.exists(local_nc):
            latest_drive = get_latest_drive_backup(DRIVE_OUT_DIR)
            if latest_drive:
                log.info(f"Restoring latest backup from Drive: {os.path.basename(latest_drive)}...")
                shutil.copy2(latest_drive, local_nc)
    else:
        files = sorted([f for f in os.listdir(LOCAL_NC) if f.startswith('smap_ogallala_')])
        local_nc = os.path.join(LOCAL_NC, files[-1]) if files else None

    start_dt = datetime(2015, 3, 31)
    cur_nc = local_nc

    if local_nc and os.path.exists(local_nc):
        try:
            with nc4.Dataset(local_nc, 'r') as ds:
                n_times = len(ds.variables['time'])
                if n_times > 0:
                    last_val = float(ds.variables['time'][-1])
                    last_dt = nc4.num2date(last_val, ds.variables['time'].units, ds.variables['time'].calendar)
                    last_dt = datetime(last_dt.year, last_dt.month, last_dt.day, last_dt.hour, last_dt.minute)
                    start_dt = last_dt + timedelta(hours=3)
                    log.info(f"Resuming {cur_nc} at {start_dt} (Existing timesteps: {n_times})")
                else:
                    raise ValueError("Placeholder file with 0 timesteps detected.")
        except Exception as e:
            log.warning(f"Existing file is empty or corrupt ({e}). Deleting and starting fresh.")
            os.remove(local_nc)
            cur_nc = local_nc
            create_nc(cur_nc, lat_sub, lon_sub, mask_2d)
    else:
        cur_nc = os.path.join(LOCAL_NC, 'SPL4SMGP_Ogallala_FULL.nc') if SINGLE_FILE_MODE else os.path.join(LOCAL_NC, 'smap_ogallala_0001.nc')
        create_nc(cur_nc, lat_sub, lon_sub, mask_2d)

    # ─── RUNTIME VARIABLES ───
    script_start_time = time.time()
    runtime_checkpoints = 0
    next_runtime_checkpoint = script_start_time + (1 * 3600) # 3 hours in seconds

    # ─── MAIN LOOP ───
    while start_dt < datetime.now():
        end_dt = min(start_dt + timedelta(days=BATCH_DATES), datetime.now())
        log.info(f"\nBatch: {start_dt.date()} to {end_dt.date()}")

        results = earthaccess.search_data(short_name=SHORT_NAME,
                                          temporal=(start_dt.strftime('%Y-%m-%dT%H:%M:%SZ'), end_dt.strftime('%Y-%m-%dT%H:%M:%SZ')))
        downloaded = earthaccess.download(results, local_path=LOCAL_WORK, threads=N_THREADS)

        # ─── BATCH VALIDATION & REDOWNLOAD LOGIC ───
        current_batch = []
        for f in downloaded:
            dt = parse_dt(str(f))
            if dt:
                current_batch.append((dt, str(f)))

        valid_granules = []
        retry_attempt = 0

        # Loop: check all -> list corrupts -> delete -> redownload -> repeat
        while current_batch and retry_attempt < MAX_RETRIES:
            valid = []
            corrupt_dts = []

            log.info(f"Validation Pass {retry_attempt+1}: Checking {len(current_batch)} files...")
            for dt, h5_path in current_batch:
                if validate_h5(h5_path):
                    valid.append((dt, h5_path))
                else:
                    corrupt_dts.append(dt)
                    try:
                        os.remove(h5_path) # Delete corrupted file
                    except OSError:
                        pass

            valid_granules.extend(valid)
            current_batch = [] # Reset for the next loop (will be populated with redownloads)

            if not corrupt_dts:
                log.info("All files validated successfully.")
                break

            retry_attempt += 1
            log.warning(f"Found {len(corrupt_dts)} corrupted files. Redownloading (Attempt {retry_attempt}/{MAX_RETRIES})...")

            # Redownload the corrupted files specifically
            for dt in corrupt_dts:
                try:
                    # Use exact temporal search to fetch the specific granule
                    res = earthaccess.search_data(short_name=SHORT_NAME,
                                                  temporal=(dt - timedelta(minutes=10), dt + timedelta(minutes=10)))
                    if res:
                        files = earthaccess.download(res, local_path=LOCAL_WORK, threads=1)
                        for f in files:
                            if parse_dt(str(f)) == dt:
                                current_batch.append((dt, str(f)))
                                break
                except Exception as e:
                    log.error(f"Failed to redownload {dt}: {e}")

        if retry_attempt == MAX_RETRIES and current_batch:
            log.error(f"Max retries reached. {len(current_batch)} files remain corrupted and will be skipped.")

        # Sort the final valid list chronologically
        valid_granules.sort(key=lambda x: x[0])
        log.info(f"Proceeding to extract {len(valid_granules)} valid granules.")

        # ─── EXTRACTION & NETCDF WRITING ───
        ds = nc4.Dataset(cur_nc, 'a')
        dates_since_flush = 0
        last_date_str = None
        processed_count = 0

        for dt, h5_path in valid_granules:
            if dt < start_dt: continue

            try:
                data = extract_granule(h5_path, row_slice, col_slice, mask_2d)
                t_idx = len(ds.variables['time'])
                t_num = nc4.date2num(dt, ds.variables['time'].units, ds.variables['time'].calendar)

                ds.variables['time'][t_idx] = t_num
                for v, arr in data.items():
                    ds.variables[v][t_idx, :, :] = arr

                processed_count += 1

                # AGGRESSIVE SYNC: Write to disk every 5 files to prevent empty placeholders
                if processed_count % 5 == 0:
                    ds.sync()

                # ─── 3-HOUR RUNTIME CHECKPOINT LOGIC ───
                current_runtime = time.time()
                if current_runtime >= next_runtime_checkpoint:
                    runtime_checkpoints += 1
                    checkpoint_name = f'SMAP_ogallala_{runtime_checkpoints * 3}hr.nc'
                    checkpoint_path = os.path.join(DRIVE_OUT_DIR, checkpoint_name)

                    log.info(f"3-hour runtime checkpoint reached. Saving {checkpoint_name} to Drive...")
                    ds.sync()
                    ds.close()
                    try:
                        robust_drive_copy(cur_nc, checkpoint_path)
                    except Exception as e:
                        log.error(f"Runtime checkpoint failed: {e}. Continuing processing.")
                    ds = nc4.Dataset(cur_nc, 'a') # Reopen dataset

                    next_runtime_checkpoint = current_runtime + (3 * 3600)
                    gc.collect()

                cur_date = dt.strftime('%Y%m%d')
                if cur_date != last_date_str:
                    dates_since_flush += 1
                    last_date_str = cur_date

                if dates_since_flush >= FLUSH_DATES:
                    ds.sync()
                    log.info(f"Flushed {processed_count} timesteps to local disk.")
                    if SINGLE_FILE_MODE and dates_since_flush >= BACKUP_DATES:
                        log.info(f"Backing up monolithic file to Drive ({t_idx} timesteps)...")
                        ds.close()
                        robust_drive_copy(cur_nc, os.path.join(DRIVE_OUT_DIR, os.path.basename(cur_nc)))
                        ds = nc4.Dataset(cur_nc, 'a')
                        dates_since_flush = 0
                    else:
                        dates_since_flush = 0
                    gc.collect()

                os.remove(h5_path)
                del data

            except Exception as e:
                log.error(f"UNEXPECTED ERROR processing {dt}: {e}. Skipping this granule.")
                if os.path.exists(h5_path):
                    try: os.remove(h5_path)
                    except: pass
                continue

        ds.close()

        log.info(f"Batch complete. Processed {processed_count} timesteps. Backing up to Drive...")
        robust_drive_copy(cur_nc, os.path.join(DRIVE_OUT_DIR, os.path.basename(cur_nc)))

        if not SINGLE_FILE_MODE:
            batch_num = int(os.path.basename(cur_nc).split('_')[-1].split('.')[0]) + 1
            cur_nc = os.path.join(LOCAL_NC, f'smap_ogallala_{batch_num:04d}.nc')
            create_nc(cur_nc, lat_sub, lon_sub, mask_2d)

        # Cleanup stragglers and advance time
        for f in glob.glob(os.path.join(LOCAL_WORK, '*.h5')): os.remove(f)
        start_dt = end_dt + timedelta(hours=3)
        gc.collect()

if __name__ == '__main__':
    run_pipeline()

datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.7.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.7.0 which is incompatible.
Enter your Earthdata Login username: watcher69
Enter your Earthdata password: ··········


/tmp/ipykernel_1269/784754619.py:90: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  roi = gdf.geometry.unary_union
/usr/local/lib/python3.12/dist-packages/earthaccess/results.py:348: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()
/usr/local/lib/python3.12/dist-packages/earthaccess/store.py:838: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum(granule.size() for granule in granules) / 1024, 2)


QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

...

PROCESSING TASKS | :   0%|          | 0/3603 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/3603 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1116 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1116 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1116 [00:00<?, ?it/s]